# AI-IDS — Model Evaluation

**Purpose:** Chapter 4 (Results & Evaluation) — quantify the performance of the trained XGBoost classifier on the held-out test set.

**Sections**
1. Setup & load model artifacts
2. Re-run test split (reproducible — same seed as training)
3. Classification report
4. Confusion matrix
5. ROC curves
6. Feature importance (trained model)
7. SHAP explanations (sample alerts)
8. Inference latency benchmark
9. Model comparison summary

All figures saved to `../docs/figures/` for the report.

---
## 1. Setup & load model artifacts

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import label_binarize

import config
from src.preprocessing.loader import DataLoader, generate_synthetic_data
from src.preprocessing.features import FeatureEngineer

FIGURES_DIR = Path('../docs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# Load trained artifacts
model  = joblib.load(config.XGB_MODEL_FILE)
scaler = joblib.load(config.SCALER_FILE)

print(f'Model:  {type(model).__name__}')
print(f'Scaler: {type(scaler).__name__}')
print(f'Classes: {[config.LABEL_NAMES[c] for c in model.classes_]}')

---
## 2. Reproduce the test split

Uses the same `RANDOM_SEED` and `TEST_SIZE` as `train_pipeline.py` so the test set is identical to the one used during training.

In [ ]:
from sklearn.model_selection import train_test_split

loader = DataLoader()
try:
    df = loader.load_cicids()
except FileNotFoundError:
    print('Real dataset not found — using synthetic data.')
    df = generate_synthetic_data(50_000)

# Encode labels (same as train_pipeline.py)
df = df[df[config.LABEL_COLUMN].isin(config.ATTACK_LABELS)].copy()
df['label'] = df[config.LABEL_COLUMN].map(config.ATTACK_LABELS)

fe = FeatureEngineer()
X = fe.select_features(df).values
y = df['label'].values

_, X_test_raw, _, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_SEED,
    stratify=y,
)

X_test = np.clip(scaler.transform(X_test_raw), -5.0, 5.0)

print(f'Test set: {X_test.shape[0]:,} samples')
print(pd.Series(y_test).map(config.LABEL_NAMES).value_counts().to_string())

---
## 3. Classification report

Precision, recall, and F1 per class. The primary metric for a multi-class imbalanced problem is **macro F1** (treats all classes equally regardless of size).

In [ ]:
t0      = time.perf_counter()
y_pred  = model.predict(X_test)
t_pred  = time.perf_counter() - t0

acc         = accuracy_score(y_test, y_pred)
macro_f1    = f1_score(y_test, y_pred, average='macro',    zero_division=0)
weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
ms_per      = 1000 * t_pred / len(X_test)

all_labels   = sorted(set(np.unique(y_test)) | set(np.unique(y_pred)))
target_names = [config.LABEL_NAMES.get(i, f'CLS_{i}') for i in all_labels]

print(f'Accuracy      : {acc:.4f}')
print(f'Macro F1      : {macro_f1:.4f}   ← primary metric')
print(f'Weighted F1   : {weighted_f1:.4f}')
print(f'Inference     : {ms_per:.3f} ms / sample')
print()
print(classification_report(y_test, y_pred,
                             labels=all_labels,
                             target_names=target_names,
                             zero_division=0))

# False Positive Rate on BENIGN
benign_id    = config.ATTACK_LABELS[config.BENIGN_LABEL]
benign_mask  = y_test == benign_id
fpr_benign   = (y_pred[benign_mask] != benign_id).mean()
print(f'FPR on Normal Traffic: {fpr_benign:.4f}  ({fpr_benign:.2%})')

---
## 4. Confusion matrix

In [ ]:
labels      = sorted(np.unique(y_test))
class_names = [config.LABEL_NAMES.get(i, f'CLS_{i}') for i in labels]

cm      = confusion_matrix(y_test, y_pred, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, data, fmt, title in [
    (axes[0], cm_norm, '.2f', 'Normalised (proportion)'),
    (axes[1], cm,      'd',   'Raw counts'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual',    fontsize=11)
    ax.set_title(f'Confusion Matrix — {title}', fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=9)

plt.suptitle(f'XGBoost — Confusion Matrix  (macro F1 = {macro_f1:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_xgb_eval.png')
plt.show()
print('Saved → confusion_matrix_xgb_eval.png')

---
## 5. ROC curves (one-vs-rest per class)

In [ ]:
y_proba  = model.predict_proba(X_test)
classes  = sorted(np.unique(y_test))
y_bin    = label_binarize(y_test, classes=classes)
if y_bin.shape[1] == 1:
    y_bin = np.hstack([1 - y_bin, y_bin])

proba_map = {c: i for i, c in enumerate(model.classes_)}

fig, ax = plt.subplots(figsize=(10, 8))
palette  = plt.cm.tab10(np.linspace(0, 1, len(classes)))

auc_scores = {}
for i, cls in enumerate(classes):
    if cls not in proba_map:
        continue
    col  = proba_map[cls]
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, col])
    try:
        auc = roc_auc_score(y_bin[:, i], y_proba[:, col])
    except ValueError:
        auc = float('nan')
    name = config.LABEL_NAMES.get(cls, f'CLS_{cls}')
    auc_scores[name] = auc
    ax.plot(fpr, tpr, lw=1.8, color=palette[i], label=f'{name}  (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.set_title('ROC Curves — XGBoost (one-vs-rest per class)', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_curves_xgb_eval.png')
plt.show()
print('Saved → roc_curves_xgb_eval.png')

print('\nAUC scores per class:')
for name, auc in sorted(auc_scores.items(), key=lambda x: -x[1]):
    print(f'  {name:<20} {auc:.4f}')
print(f'  Mean AUC: {np.nanmean(list(auc_scores.values())):.4f}')

---
## 6. Feature importance (trained XGBoost)

Feature importances from the actual trained model — more reliable than the quick-RF estimate in the EDA notebook.

In [ ]:
imp_df = pd.read_csv(config.FEATURE_IMPORTANCE_FILE)
imp_df = imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
colors  = ['#c0392b' if imp_df['importance'].iloc[-1] == v else '#2980b9'
           for v in imp_df['importance']]
ax.barh(imp_df['feature'], imp_df['importance'], color=colors, edgecolor='none')
ax.set_xlabel('Feature importance (gain)', fontsize=11)
ax.set_title('XGBoost Feature Importance — trained on full CICIDS2017', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance_xgb.png')
plt.show()
print('Saved → feature_importance_xgb.png')
print()
print('Top 10 features:')
print(imp_df.sort_values('importance', ascending=False).head(10).to_string(index=False))

---
## 7. SHAP explanations — sample alerts

For 5 randomly sampled attack predictions, compute SHAP values showing which features drove the detection decision. This is the per-alert explainability shown in the dashboard.

In [ ]:
try:
    import shap
    from src.explainability.shap_explainer import SHAPExplainer

    explainer = SHAPExplainer(model, config.SELECTED_FEATURES)

    # Pick 5 attack predictions for illustration
    attack_mask = y_pred != config.ATTACK_LABELS[config.BENIGN_LABEL]
    sample_idx  = np.where(attack_mask)[0][:5]

    print('SHAP top-5 feature contributions for 5 sample attack detections:')
    print('=' * 60)
    for idx in sample_idx:
        pred_class = int(y_pred[idx])
        contribs   = explainer.explain(X_test[idx:idx+1], class_idx=pred_class)
        print(f'\n  Sample {idx} → predicted: {config.LABEL_NAMES.get(pred_class)}')
        for feat, val in contribs.items():
            bar = '█' * min(int(abs(val) * 30), 30)
            sign = '+' if val >= 0 else ''
            print(f'    {feat:<32} {sign}{val:+.4f}  {bar}')

except ImportError:
    print('shap not installed — run: pip install shap>=0.47')

---
## 8. Inference latency benchmark

Measures single-sample inference latency — the key real-time constraint. The IDS must classify each network flow faster than it arrives.

In [ ]:
N_WARMUP = 20
N_BENCH  = 500

# Warm up (avoids JIT / cache-cold effects)
for i in range(N_WARMUP):
    model.predict(X_test[i:i+1])

latencies = []
for i in range(N_BENCH):
    t0 = time.perf_counter()
    model.predict(X_test[i % len(X_test):i % len(X_test) + 1])
    latencies.append((time.perf_counter() - t0) * 1000)  # ms

lats = np.array(latencies)
print(f'Single-sample inference latency ({N_BENCH} runs):')
print(f'  Mean   : {lats.mean():.3f} ms')
print(f'  Median : {np.median(lats):.3f} ms')
print(f'  P95    : {np.percentile(lats, 95):.3f} ms')
print(f'  P99    : {np.percentile(lats, 99):.3f} ms')
print(f'  Max    : {lats.max():.3f} ms')
print(f'\nEstimated max throughput: {1000/lats.mean():.0f} flows/sec')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lats, bins=40, color='steelblue', edgecolor='none', alpha=0.85)
ax.axvline(lats.mean(),       color='red',    lw=1.5, linestyle='--', label=f'Mean {lats.mean():.2f} ms')
ax.axvline(np.percentile(lats, 95), color='orange', lw=1.5, linestyle=':', label=f'P95 {np.percentile(lats,95):.2f} ms')
ax.set_xlabel('Inference time (ms)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Single-sample inference latency — XGBoost', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'inference_latency.png')
plt.show()
print('Saved → inference_latency.png')

---
## 9. Summary table for the report

In [ ]:
summary = pd.DataFrame([{
    'Model':        'XGBoost (XGBClassifier)',
    'Accuracy':     f'{acc:.4f}',
    'Macro F1':     f'{macro_f1:.4f}',
    'Weighted F1':  f'{weighted_f1:.4f}',
    'FPR (benign)': f'{fpr_benign:.4f}',
    'Mean AUC':     f'{np.nanmean(list(auc_scores.values())):.4f}',
    'Latency (ms)': f'{lats.mean():.3f}',
}])
print(summary.to_string(index=False))
summary

---
## Figures produced

| Figure | Report section |
|---|---|
| `confusion_matrix_xgb_eval.png` | Chapter 4 — Results |
| `roc_curves_xgb_eval.png` | Chapter 4 — Results |
| `feature_importance_xgb.png` | Chapter 4 — Feature importance |
| `inference_latency.png` | Chapter 4 — Real-time performance |